# Agregar Variables de Congruencia a Bases Definitivas

Este notebook agrega las variables de congruencia/incongruencia ideológica calculadas a las bases definitivas.

## Variables que se agregan:

- `CO_Congruente`: Cambio de opinión ideológicamente consistente (Progresistas→Izq + Conservadores→Der)
- `CO_Incongruente`: Cambio de opinión ideológicamente inconsistente (Progresistas→Der + Conservadores→Izq)
- `CT_Congruente`: Cambio de tiempo ideológicamente consistente (Progresistas→Izq + Conservadores→Der)
- `CT_Incongruente`: Cambio de tiempo ideológicamente inconsistente (Progresistas→Der + Conservadores→Izq)

## Fuente:

Código basado en el notebook `52. Análisis Congruencia Ideológica.ipynb` de la carpeta Código.

In [ ]:
import pandas as pd
import numpy as np
import os

print("✓ Librerías cargadas exitosamente")

## 1. Cargar Bases Definitivas

In [ ]:
# Rutas a los archivos Excel.
Ruta_Base = os.path.join(
    os.getcwd(), 
    '..', 
    'Data', 
    'Bases definitivas'
)

Excel_Generales = os.path.join(Ruta_Base, 'Generales.xlsx')
Excel_Ballotage = os.path.join(Ruta_Base, 'Ballotage.xlsx')

# Cargar DataFrames desde Excel.
df_Generales = pd.read_excel(Excel_Generales)
df_Ballotage = pd.read_excel(Excel_Ballotage)

dfs_Finales = {
    'Generales': df_Generales,
    'Ballotage': df_Ballotage
}

print(f"✓ Datos cargados desde Excel:")
print(f"  - Generales: {len(df_Generales)} registros")
print(f"  - Ballotage: {len(df_Ballotage)} registros")

## 2. Verificar Variables Necesarias

In [ ]:
# Variables necesarias para CO.
Vars_Necesarias_CO = [
    'Cambio_Op_Sum_Pro_Izq',
    'Cambio_Op_Sum_Pro_Der',
    'Cambio_Op_Sum_Con_Izq',
    'Cambio_Op_Sum_Con_Der'
]

# Variables necesarias para CT.
Vars_Necesarias_CT = [
    'Cambio_Tiempo_Sum_Pro_Izq',
    'Cambio_Tiempo_Sum_Pro_Der',
    'Cambio_Tiempo_Sum_Con_Izq',
    'Cambio_Tiempo_Sum_Con_Der'
]

print("Verificando variables necesarias:\n")

for Nombre_DF, df in dfs_Finales.items():
    print(f"{Nombre_DF}:")
    
    # Verificar CO.
    Faltantes_CO = [
        v for v in Vars_Necesarias_CO if v not in df.columns
    ]
    if Faltantes_CO:
        print(f"  ⚠️  Variables CO faltantes: {Faltantes_CO}")
    else:
        print(f"  ✓ Todas las variables CO presentes")
    
    # Verificar CT.
    Faltantes_CT = [
        v for v in Vars_Necesarias_CT if v not in df.columns
    ]
    if Faltantes_CT:
        print(f"  ⚠️  Variables CT faltantes: {Faltantes_CT}")
    else:
        print(f"  ✓ Todas las variables CT presentes")
    
    print()

## 3. Crear Variables de Congruencia e Incongruencia

In [ ]:
print("Creando variables de Congruencia e Incongruencia "
      "(usando PROMEDIOS)...\n")

# Definir ítems progresistas y conservadores.
Items_Progresistas = [
    5, 6, 9, 11, 16, 20, 24, 25, 27, 28
]
Items_Conservadores = [
    3, 4, 7, 8, 10, 19, 22, 23, 29, 30
]

for Nombre_DF, df in dfs_Finales.items():
    print(f"Procesando {Nombre_DF}...")
    
    # ========================================================
    # CONGRUENTE CO
    # ========================================================
    
    # Necesitamos: (Pro→Izq + Con→Der) / n_items_total.
    
    # Contar ítems progresistas hacia izquierda.
    Cols_Pro_Izq = [
        f'CO_Item_{i}_Izq' for i in Items_Progresistas
    ]
    N_Pro_Izq = df[Cols_Pro_Izq].notna().sum(axis=1)
    
    # Contar ítems conservadores hacia derecha.
    Cols_Con_Der = [
        f'CO_Item_{i}_Der' for i in Items_Conservadores
    ]
    N_Con_Der = df[Cols_Con_Der].notna().sum(axis=1)
    
    # Total de ítems congruentes.
    N_Congruente_CO = N_Pro_Izq + N_Con_Der
    
    # Suma y promedio.
    Suma_Congruente_CO = (
        df['Cambio_Op_Sum_Pro_Izq'] + 
        df['Cambio_Op_Sum_Con_Der']
    )
    df['CO_Congruente'] = np.where(
        N_Congruente_CO > 0, 
        Suma_Congruente_CO / N_Congruente_CO, 
        np.nan
    )
    print(f"  ✓ CO_Congruente creada (promedio)")
    
    # ========================================================
    # INCONGRUENTE CO
    # ========================================================
    
    # Necesitamos: (Pro→Der + Con→Izq) / n_items_total.
    
    # Contar ítems progresistas hacia derecha.
    Cols_Pro_Der = [
        f'CO_Item_{i}_Der' for i in Items_Progresistas
    ]
    N_Pro_Der = df[Cols_Pro_Der].notna().sum(axis=1)
    
    # Contar ítems conservadores hacia izquierda.
    Cols_Con_Izq = [
        f'CO_Item_{i}_Izq' for i in Items_Conservadores
    ]
    N_Con_Izq = df[Cols_Con_Izq].notna().sum(axis=1)
    
    # Total de ítems incongruentes.
    N_Incongruente_CO = N_Pro_Der + N_Con_Izq
    
    # Suma y promedio.
    Suma_Incongruente_CO = (
        df['Cambio_Op_Sum_Pro_Der'] + 
        df['Cambio_Op_Sum_Con_Izq']
    )
    df['CO_Incongruente'] = np.where(
        N_Incongruente_CO > 0,
        Suma_Incongruente_CO / N_Incongruente_CO,
        np.nan
    )
    print(f"  ✓ CO_Incongruente creada (promedio)")
    
    # ========================================================
    # CONGRUENTE CT
    # ========================================================
    
    # Contar ítems progresistas hacia izquierda (CT).
    Cols_CT_Pro_Izq = [
        f'CT_Item_{i}_Izq' for i in Items_Progresistas
    ]
    N_CT_Pro_Izq = df[Cols_CT_Pro_Izq].notna().sum(axis=1)
    
    # Contar ítems conservadores hacia derecha (CT).
    Cols_CT_Con_Der = [
        f'CT_Item_{i}_Der' for i in Items_Conservadores
    ]
    N_CT_Con_Der = df[Cols_CT_Con_Der].notna().sum(axis=1)
    
    # Total de ítems congruentes CT.
    N_Congruente_CT = N_CT_Pro_Izq + N_CT_Con_Der
    
    # Suma y promedio.
    Suma_Congruente_CT = (
        df['Cambio_Tiempo_Sum_Pro_Izq'] + 
        df['Cambio_Tiempo_Sum_Con_Der']
    )
    df['CT_Congruente'] = np.where(
        N_Congruente_CT > 0,
        Suma_Congruente_CT / N_Congruente_CT,
        np.nan
    )
    print(f"  ✓ CT_Congruente creada (promedio)")
    
    # ========================================================
    # INCONGRUENTE CT
    # ========================================================
    
    # Contar ítems progresistas hacia derecha (CT).
    Cols_CT_Pro_Der = [
        f'CT_Item_{i}_Der' for i in Items_Progresistas
    ]
    N_CT_Pro_Der = df[Cols_CT_Pro_Der].notna().sum(axis=1)
    
    # Contar ítems conservadores hacia izquierda (CT).
    Cols_CT_Con_Izq = [
        f'CT_Item_{i}_Izq' for i in Items_Conservadores
    ]
    N_CT_Con_Izq = df[Cols_CT_Con_Izq].notna().sum(axis=1)
    
    # Total de ítems incongruentes CT.
    N_Incongruente_CT = N_CT_Pro_Der + N_CT_Con_Izq
    
    # Suma y promedio.
    Suma_Incongruente_CT = (
        df['Cambio_Tiempo_Sum_Pro_Der'] + 
        df['Cambio_Tiempo_Sum_Con_Izq']
    )
    df['CT_Incongruente'] = np.where(
        N_Incongruente_CT > 0,
        Suma_Incongruente_CT / N_Incongruente_CT,
        np.nan
    )
    print(f"  ✓ CT_Incongruente creada (promedio)")
    
    print()

print("✅ Variables de Congruencia/Incongruencia creadas "
      "exitosamente (PROMEDIOS)")

## 4. Verificar Estadísticas Descriptivas

In [ ]:
print("="*70)
print("ESTADÍSTICAS DESCRIPTIVAS - VARIABLES DE CONGRUENCIA")
print("="*70)

Variables_Analizar = [
    'CO_Congruente', 
    'CO_Incongruente', 
    'CT_Congruente', 
    'CT_Incongruente'
]

for Nombre_DF, df in dfs_Finales.items():
    print(f"\n📊 {Nombre_DF}:")
    print("\n" + "-"*70)
    
    for var in Variables_Analizar:
        if var in df.columns:
            datos = df[var].dropna()
            
            if len(datos) > 0:
                print(f"\n{var}:")
                print(f"  n = {len(datos)}")
                print(f"  Media = {datos.mean():.4f}")
                print(f"  Mediana = {datos.median():.4f}")
                print(f"  DE = {datos.std():.4f}")
                print(f"  Min = {datos.min():.4f}")
                print(f"  Max = {datos.max():.4f}")
            else:
                print(f"\n{var}: Sin datos")
        else:
            print(f"\n{var}: Variable no encontrada")
    
    print("\n" + "-"*70)

print("\n" + "="*70)

## 5. Guardar Bases Actualizadas

In [ ]:
print("Guardando bases definitivas actualizadas...\n")

for Nombre_DF, df in dfs_Finales.items():
    # Guardar en la misma ubicación, 
    # sobreescribiendo archivo original.
    Archivo = f'{Nombre_DF}.xlsx'
    Ruta = os.path.join(Ruta_Base, Archivo)
    
    # Hacer backup antes de sobreescribir.
    Ruta_Backup = os.path.join(
        Ruta_Base, 
        f'{Nombre_DF}_backup.xlsx'
    )
    
    # Si existe el archivo original, hacer backup.
    if os.path.exists(Ruta):
        import shutil
        shutil.copy2(Ruta, Ruta_Backup)
        print(f"✓ Backup creado: {Archivo}_backup.xlsx")
    
    # Guardar archivo actualizado.
    df.to_excel(Ruta, index=False)
    print(f"✓ {Archivo} actualizado con variables de "
          f"congruencia")
    print(f"  Total de columnas: {len(df.columns)}")
    print(f"  Total de registros: {len(df)}")
    print()

print("✅ Todas las bases definitivas actualizadas "
      "exitosamente")

## 6. Resumen Final

In [ ]:
print("="*70)
print("RESUMEN FINAL")
print("="*70)

print("\n📊 Variables agregadas a las bases definitivas:")
print("  - CO_Congruente (Progresistas→Izq + "
      "Conservadores→Der)")
print("  - CO_Incongruente (Progresistas→Der + "
      "Conservadores→Izq)")
print("  - CT_Congruente (Progresistas→Izq + "
      "Conservadores→Der)")
print("  - CT_Incongruente (Progresistas→Der + "
      "Conservadores→Izq)")

print("\n📁 Archivos actualizados:")
print("  - Generales.xlsx (con backup)")
print("  - Ballotage.xlsx (con backup)")

print("\n🔍 Ítems utilizados:")
print(f"  - Progresistas: {Items_Progresistas}")
print(f"  - Conservadores: {Items_Conservadores}")

print("\n🎯 Interpretación:")
print("  - Congruente: Cambios ideológicamente consistentes")
print("  - Incongruente: Cambios ideológicamente "
      "inconsistentes")
print("  - Valores calculados como PROMEDIOS por sujeto")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)